# ⚡ Tutorial & Panduan: Training YOLO-X / YOLO11-X di Google Colab
**Dataset:** `tire_demage_20260917_002759`  
**Target Model:** ⚡ **YOLO-X / YOLO11-X** (High-Performance Real-Time Object Detection)  
**Pretrained Base:** `yolo11x.pt` | **Epochs:** 200 | **Optimizer:** `AdamW` | **Batch:** 16  
**Target Hardware:** NVIDIA T4 GPU (Google Colab Free Tier)  
**Fitur Unggulan:** 🔄 **Auto-Resume Epoch** & 💾 **Google Drive Persistent Cache**  

---
### 🛡️ Solusi Anti-Terputus (T4 Free Disconnect & Auto-Resume):
Google Colab Free Tier dapat terputus sewaktu-waktu. Notebook ini dilengkapi sistem **Auto-Resume** dan **Penyimpanan Google Drive**:
- **Checkpoint Otomatis (`last.pt`):** Disimpan di Google Drive setiap 5 epoch dan di setiap epoch. Jika runtime Colab terputus di epoch 45, saat dijalankan lagi akan **otomatis melanjutkan ke epoch 46** tanpa mengulang dari 0!
- **Cache Dataset Cepat:** Dataset yang telah diunduh otomatis disimpan di Google Drive (`dataset_tire_demage_20260917_002759.zip`). Sesi berikutnya hanya butuh **10 detik** untuk memulihkan dataset tanpa download ulang!
- **Download Otomatis:** Setelah training selesai, file bobot (`best.pt`) dan evaluasi (.zip) otomatis diunduh ke komputer Anda untuk diunggah ke web **Raray Vision**.

### 🖥️ Langkah 0: Cek Akses GPU (Runtime Check)
Memastikan runtime Google Colab menggunakan **GPU Tesla T4** agar proses training berjalan cepat.

In [ ]:
# Cek apakah GPU tersedia menggunakan PyTorch dan command nvidia-smi
import torch
import subprocess

gpu_available = torch.cuda.is_available()
print('=== STATUS RUNTIME GOOGLE COLAB ===')
if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    print(f'[OK] GPU Terdeteksi: {device_name}')
    print('\nDetail Spesifikasi GPU:')
    try:
        gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode('utf-8')
        print(gpu_info)
    except Exception:
        print('Tidak dapat memanggil nvidia-smi, namun GPU CUDA tetap aktif.')
else:
    print('[WARNING] GPU tidak terdeteksi! Silakan ganti runtime ke T4 GPU via menu: Runtime > Change runtime type > T4 GPU.')


### 💾 Langkah 0B: Hubungkan Google Drive (Penyimpanan Permanen & Anti-Disconnect)
**Langkah ini sangat penting untuk Colab Free Tier!**
Menyambungkan Google Drive Anda agar checkpoint (`last.pt`) dan cache dataset tersimpan permanen.
Jika runtime Google Colab terputus di tengah jalan, Anda tidak akan kehilangan progres training sama sekali.

In [ ]:
# Mount Google Drive untuk penyimpanan permanen checkpoint & cache dataset
import os

USE_GOOGLE_DRIVE = True  # Ubah ke False jika HANYA ingin menyimpan di disk sementara Colab
DRIVE_BASE_DIR = '/content/drive/MyDrive/raray_vision_colab'
DRIVE_RUNS_DIR = os.path.join(DRIVE_BASE_DIR, 'runs')
DRIVE_DATASET_ZIP = os.path.join(DRIVE_BASE_DIR, f'dataset_{folder_name}.zip')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        print('Menghubungkan ke Google Drive...')
        drive.mount('/content/drive')
        os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)
        print(f'[OK] Google Drive terhubung! Folder proyek: {DRIVE_BASE_DIR}')
        print(f'[OK] Folder Checkpoint Runs: {DRIVE_RUNS_DIR}')
    except Exception as e:
        print(f'[WARNING] Google Drive tidak tersambung ({e}). Menggunakan penyimpanan lokal Colab.')
        USE_GOOGLE_DRIVE = False
        DRIVE_RUNS_DIR = 'raray_vision_runs'
else:
    print('Google Drive dinonaktifkan. Menggunakan penyimpanan lokal Colab.')
    DRIVE_RUNS_DIR = 'raray_vision_runs'


### 📦 Langkah 1: Instalasi Library & Dependensi
Menginstal library resmi yang diperlukan: **Ultralytics** (YOLO11, YOLO-X, YOLO-26, RT-DETR), **PyYAML**, dan library pendukung.

In [ ]:
# Install library Ultralytics dan dependensi pendukung
!pip install -q --upgrade ultralytics pyyaml requests tqdm pillow

import ultralytics
print(f'[OK] Ultralytics Version: {ultralytics.__version__}')
ultralytics.checks()


### 📁 Langkah 2: Menyiapkan Struktur Direktori & Cek Cache Dataset di Google Drive
Menyiapkan folder lokal di Colab (`/content/dataset/images/train`, `/content/dataset/images/val`, dll) dan memeriksa apakah dataset sudah pernah disimpan di Google Drive.
**Jika sudah ada di Google Drive**, dataset akan diekstrak langsung dalam ~10 detik tanpa perlu download ulang!

In [ ]:
import os, glob, shutil, time, json, zipfile

# Inisialisasi struktur direktori YOLO di Colab
base_dir = os.path.abspath('dataset')
train_img_dir = os.path.join(base_dir, 'images', 'train')
val_img_dir = os.path.join(base_dir, 'images', 'val')
train_lbl_dir = os.path.join(base_dir, 'labels', 'train')
val_lbl_dir = os.path.join(base_dir, 'labels', 'val')

for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

DATASET_RESTORED_FROM_CACHE = False

# Cek apakah arsip dataset sudah tersimpan di Google Drive
if 'DRIVE_DATASET_ZIP' in locals() and os.path.exists(DRIVE_DATASET_ZIP):
    print(f'[OK] DITEMUKAN CACHE DATASET DI GOOGLE DRIVE: {DRIVE_DATASET_ZIP}')
    print('Mengekstrak dataset langsung ke Colab (~10 detik, anti-download ulang)...')
    with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'r') as zf:
        zf.extractall('/content')
    
    train_imgs = len(glob.glob(os.path.join(train_img_dir, '*.*')))
    val_imgs = len(glob.glob(os.path.join(val_img_dir, '*.*')))
    print(f'[OK] Dataset sukses dipulihkan dari cache Drive! (Train: {train_imgs}, Val: {val_imgs} gambar)')
    if train_imgs > 0:
        DATASET_RESTORED_FROM_CACHE = True
else:
    print('[INFO] Cache dataset di Google Drive belum ada. Sistem akan mengunduh dan menyimpannya otomatis di Langkah 6.')


### 🛠️ Langkah 3: Inisialisasi Download Helper (Anti-Timeout, Chunking & Auto-Retry)
Menyiapkan helper download dengan connection pooling, header browser, dan chunked streaming untuk mengatasi file gambar berukuran besar dan koneksi jaringan lambat.

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
retries = Retry(
    total=7,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=False
)
adapter = HTTPAdapter(max_retries=retries, pool_connections=20, pool_maxsize=20)
session.mount('https://', adapter)
session.mount('http://', adapter)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def robust_download_file(url, target_path, timeout=(15, 60)):
    try:
        resp = session.get(url, headers=HEADERS, timeout=timeout, stream=True)
        if resp.status_code == 200:
            with open(target_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=131072):
                    if chunk:
                        f.write(chunk)
            return True
        return False
    except Exception as e:
        return False

print('[OK] Helper download robust siap digunakan.')


### ⚙️ Langkah 4: Download & Setup Konfigurasi `data.yaml`
Mengunduh file konfigurasi `data.yaml` yang berisi daftar kelas dan path folder dataset.

In [ ]:
import yaml

yaml_url = 'https://is3.cloudhost.id/raray-vision/datasets/tire_demage_20260917_002759/data.yaml'
print(f'Mengunduh data.yaml dari: {yaml_url}')

success = robust_download_file(yaml_url, 'data.yaml', timeout=(10, 30))
if not success or not os.path.exists('data.yaml') or os.path.getsize('data.yaml') == 0:
    print('⚠ Menggunakan template data.yaml darurat...')
    fallback_yaml = {
        'path': os.path.abspath('dataset'),
        'train': 'images/train',
        'val': 'images/val',
        'names': {0: 'defect'}
    }
    with open('data.yaml', 'w') as f:
        yaml.dump(fallback_yaml, f, sort_keys=False)
else:
    with open('data.yaml', 'r') as f:
        ydata = yaml.safe_load(f)
    ydata['path'] = os.path.abspath('dataset')
    ydata['train'] = 'images/train'
    ydata['val'] = 'images/val'
    with open('data.yaml', 'w') as f:
        yaml.dump(ydata, f, sort_keys=False)

print('[OK] Konfigurasi data.yaml siap:')
with open('data.yaml', 'r') as f:
    print(f.read().strip())


### 📑 Langkah 5: Download Anotasi Dataset (Tasks JSON)
Mengambil daftar 3.500+ task anotasi (bounding box, label, dan link gambar).

In [ ]:
tasks_url = 'http://127.0.0.1:8000/api/v1/models/data/datasets/import-tasks/ds_65ee5ff493a54d6f8f53a47941fa7597.json'
print(f'Mengunduh anotasi tasks dari: {tasks_url}')

tasks_data = None
for attempt in range(5):
    try:
        r = session.get(tasks_url, headers=HEADERS, timeout=(15, 60))
        if r.status_code == 200:
            tasks_data = r.json()
            break
    except Exception as e:
        print(f'Percobaan {attempt+1} gagal: {e}. Mengulang dalam 2 detik...')
        time.sleep(2)

if tasks_data:
    print(f'[OK] Berhasil memuat {len(tasks_data)} tasks anotasi.')
else:
    print('[WARNING] Gagal mengunduh tasks_data dari API!')
    tasks_data = []


### 📥 Langkah 6: Download Gambar & Pembuatan Cache di Google Drive
Jika dataset sudah dipulihkan dari Google Drive di Langkah 2, langkah ini akan **otomatis melompat** (selesai dalam 1 detik).
Jika belum, sistem akan mengunduh gambar menggunakan multi-threading dan langsung mengarsipkannya ke Google Drive.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

if DATASET_RESTORED_FROM_CACHE:
    print('⚡ Dataset sudah lengkap dipulihkan dari cache Google Drive!')
    print('Melompati proses download gambar.')
else:
    print(f'Memulai download {len(tasks_data)} gambar...')
    
    # Build class mapping dari data.yaml
    with open('data.yaml', 'r') as f:
        yaml_cfg = yaml.safe_load(f)
    class_names = yaml_cfg.get('names', {0: 'defect'})
    if isinstance(class_names, list):
        name_to_idx = {name: idx for idx, name in enumerate(class_names)}
    elif isinstance(class_names, dict):
        name_to_idx = {name: int(idx) for idx, name in class_names.items()}
    else:
        name_to_idx = {}

    def process_task(task_idx, task):
        try:
            img_url = task.get('data', {}).get('image') or task.get('image')
            if not img_url:
                return
            is_val = (task_idx % 5 == 0)
            split = 'val' if is_val else 'train'
            
            fname = os.path.basename(img_url).split('?')[0]
            if not fname:
                fname = f'img_{task_idx}.jpg'
            ext = os.path.splitext(fname)[1] or '.jpg'
            base_name = os.path.splitext(fname)[0]
            
            img_save_path = os.path.join(base_dir, 'images', split, f'{base_name}{ext}')
            lbl_save_path = os.path.join(base_dir, 'labels', split, f'{base_name}.txt')
            
            if not os.path.exists(img_save_path) or os.path.getsize(img_save_path) == 0:
                robust_download_file(img_url, img_save_path)
            
            # Tulis label YOLO
            labels = []
            for ann in task.get('annotations', []):
                for res in ann.get('result', []):
                    if res.get('type') == 'rectanglelabels':
                        val = res.get('value', {})
                        lbl_name = val.get('rectanglelabels', ['defect'])[0]
                        cls_id = name_to_idx.get(lbl_name, 0)
                        x = (val.get('x', 0) + val.get('width', 0) / 2) / 100.0
                        y = (val.get('y', 0) + val.get('height', 0) / 2) / 100.0
                        w = val.get('width', 0) / 100.0
                        h = val.get('height', 0) / 100.0
                        labels.append(f'{cls_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
            
            with open(lbl_save_path, 'w') as lf:
                lf.write('\n'.join(labels))
        except Exception:
            pass

    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(lambda item: process_task(item[0], item[1]), enumerate(tasks_data)), total=len(tasks_data), desc='Downloading Dataset'))
    
    # Auto-Cache ke Google Drive jika terhubung
    if 'USE_GOOGLE_DRIVE' in locals() and USE_GOOGLE_DRIVE and 'DRIVE_DATASET_ZIP' in locals():
        print('\n💾 Mengarsipkan dataset ke Google Drive untuk proteksi disconnect...')
        try:
            with zipfile.ZipFile(DRIVE_DATASET_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
                for root, _, files in os.walk(base_dir):
                    for f in files:
                        full_p = os.path.join(root, f)
                        rel_p = os.path.relpath(full_p, '/content')
                        zf.write(full_p, arcname=rel_p)
            print(f'[OK] Cache dataset tersimpan permanen di Google Drive: {DRIVE_DATASET_ZIP}')
        except Exception as e:
            print(f'Gagal mengarsipkan dataset ke Drive: {e}')


### 🔍 Langkah 7: Verifikasi Integritas Dataset
Memeriksa jumlah file gambar dan label di folder train dan val untuk memastikan dataset 100% siap ditraining.

In [ ]:
train_imgs = glob.glob(os.path.join(train_img_dir, '*.*'))
val_imgs = glob.glob(os.path.join(val_img_dir, '*.*'))
train_lbls = glob.glob(os.path.join(train_lbl_dir, '*.txt'))
val_lbls = glob.glob(os.path.join(val_lbl_dir, '*.txt'))

print('=== RINGKASAN DATASET SIAP TRAINING ===')
print(f'Images Train : {len(train_imgs)} gambar')
print(f'Labels Train : {len(train_lbls)} file anotasi')
print(f'Images Val   : {len(val_imgs)} gambar')
print(f'Labels Val   : {len(val_lbls)} file anotasi')

if len(train_imgs) == 0:
    print('[WARNING] Gambar train masih kosong! Periksa koneksi atau langkah download di atas.')
else:
    print('[OK] Dataset 100% Siap untuk Training!')


### 🚀 Langkah 8: Training Model ⚡ **YOLO-X / YOLO11-X** (High-Performance Real-Time Object Detection)
YOLO-X adalah arsitektur model besar untuk akurasi tertinggi dalam deteksi objek real-time.
- **Pretrained Base:** `yolo11x.pt`
- **Epochs:** `200`
- **Optimizer:** `AdamW`
- **Batch Size:** `16`
- **🔄 Auto-Resume:** Jika Colab terputus, jalankan kembali cell ini! Sistem akan otomatis mendeteksi `last.pt` dan melanjutkan dari epoch terakhir tanpa mengulang dari 0.

In [ ]:
from ultralytics import YOLO

# Tentukan folder output penyimpanan (Google Drive jika tersambung, atau lokal Colab)
runs_output_dir = DRIVE_RUNS_DIR if ('USE_GOOGLE_DRIVE' in locals() and USE_GOOGLE_DRIVE and os.path.exists('/content/drive/MyDrive')) else 'raray_vision_runs'
run_name = 'yolo_x_200epochs'

# Cek keberadaan file checkpoint last.pt untuk Auto-Resume
ckpt_path = os.path.join(runs_output_dir, run_name, 'weights', 'last.pt')
if not os.path.exists(ckpt_path):
    local_ckpt = os.path.join('raray_vision_runs', run_name, 'weights', 'last.pt')
    if os.path.exists(local_ckpt):
        ckpt_path = local_ckpt

if os.path.exists(ckpt_path):
    print('==================================================================')
    print(f'🔄 CHECKPOINT TERAKHIR DITEMUKAN: {ckpt_path}')
    print('⚡ MELANJUTKAN TRAINING YOLOX DARI EPOCH TERAKHIR (AUTO-RESUME)...')
    print('==================================================================')
    model_yolox = YOLO(ckpt_path)
    results_yolox = model_yolox.train(resume=True)
else:
    print('🚀 MEMULAI TRAINING BARU YOLOX (200 EPOCHS)...')
    print(f'📁 Lokasi penyimpanan weights permanen: {runs_output_dir}/{run_name}')
    model_yolox = YOLO('yolo11x.pt')
    results_yolox = model_yolox.train(
        data='data.yaml',
        epochs=200,
        imgsz=640,
        batch=16,
        device=0, # GPU 0 (Tesla T4)
        workers=4,
        optimizer='AdamW',
        lr0=0.001,
        patience=50,
        save=True,
        save_period=5, # Simpan checkpoint berkala setiap 5 epoch
        project=runs_output_dir,
        name=run_name
    )

print('\n📊 VALIDASI YOLOX:')
metrics_yolox = model_yolox.val()
print('Validation mAP50:', getattr(getattr(metrics_yolox, 'box', None), 'map50', 'N/A'))

# Export ke format ONNX
model_yolox.export(format='onnx', dynamic=True, simplify=True)
print('✓ YOLOX weights & ONNX berhasil diekspor!')


### 📦 Langkah 9: Pengemasan Bobot Model & Hasil Evaluasi (YOLOX)
Tahap ini mengumpulkan file bobot terbaik (`best.pt`), bobot ONNX (`best.onnx`), kurva metrik, confusion matrix, dan `results.csv` ke dalam arsip ZIP yang rapi.
Mencari file training di folder Google Drive maupun folder lokal Colab secara otomatis.

In [ ]:
# ==========================================================
# PENGATURAN METADATA & ARSIP ZIP TRAINING
# ==========================================================
import os, glob, shutil, json, zipfile
from datetime import datetime

# 📝 Masukkan tanggal dan keterangan setting model Anda:
TANGGAL_TRAINING = datetime.now().strftime('%Y-%m-%d')  # Format: YYYY-MM-DD
KETERANGAN_SETTING = 'YOLO-X: 200 Epochs, Imgsz 640, Batch 16, Optimizer AdamW, Tesla T4 GPU'

print('=== PENGEMASAN ARSIP EVALUASI & BOBOT MODEL (YOLOX) ===')
# Cari run training di Google Drive dan raray_vision_runs lokal
search_dirs = []
if 'DRIVE_RUNS_DIR' in locals() and os.path.exists(DRIVE_RUNS_DIR):
    search_dirs.extend(glob.glob(f'{DRIVE_RUNS_DIR}/yolo_x_200epochs*'))
search_dirs.extend(glob.glob('raray_vision_runs/yolo_x_200epochs*'))
valid_runs = [r for r in search_dirs if os.path.isdir(r) and not r.endswith('.zip')]
all_runs = sorted(list(set(valid_runs)), key=os.path.getmtime, reverse=True)

if not all_runs:
    print('❌ Belum ada folder hasil training ditemukan. Silakan jalankan cell Langkah 8 terlebih dahulu.')
else:
    latest_run = all_runs[0]
    run_name = os.path.basename(latest_run)
    print(f'📁 Direktori Training Terpilih: {latest_run}')
    
    # 1. Buat file metadata setting training
    meta_info = {
        'training_date': TANGGAL_TRAINING,
        'training_settings': KETERANGAN_SETTING,
        'model_architecture': 'YOLOX',
        'run_name': run_name,
        'created_at': datetime.now().isoformat()
    }
    meta_path = os.path.join(latest_run, 'training_meta.json')
    with open(meta_path, 'w', encoding='utf-8') as mf:
        json.dump(meta_info, mf, indent=2)
    print(f'✓ Metadata setting tersimpan di: {meta_path}')
    
    # 2. Kemas Arsip Evaluasi (.zip) untuk diunggah ke Raray Vision
    eval_zip_name = f'evaluasi_yolox_{run_name}_{TANGGAL_TRAINING}.zip'
    eval_target_files = [
        'results.csv', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
        'PR_curve.png', 'F1_curve.png', 'results.png', 'labels.jpg',
        'val_batch0_pred.jpg', 'training_meta.json'
    ]
    with zipfile.ZipFile(eval_zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for ef in eval_target_files:
            fp = os.path.join(latest_run, ef)
            if os.path.exists(fp):
                zf.write(fp, arcname=ef)
                print(f'  + Arsip Eval: {ef}')
    print(f'✓ File Evaluasi ZIP siap: {eval_zip_name} ({os.path.getsize(eval_zip_name)/1024:.1f} KB)')
    
    # 3. Identifikasi bobot best.pt dan best.onnx
    best_pt_path = os.path.join(latest_run, 'weights', 'best.pt')
    best_onnx_path = os.path.join(latest_run, 'weights', 'best.onnx')
    if not os.path.exists(best_onnx_path):
        pot_onnx = glob.glob(os.path.join(latest_run, '*.onnx'))
        if pot_onnx:
            best_onnx_path = pot_onnx[0]
    
    if os.path.exists(best_pt_path):
        sz_pt = os.path.getsize(best_pt_path) / (1024 * 1024)
        print(f'✓ File Bobot Model: {best_pt_path} ({sz_pt:.2f} MB)')
    else:
        print('⚠ File best.pt belum ditemukan di folder weights.')
        
    # 4. Buat All-In-One ZIP Bundle (Weights + Evaluasi)
    bundle_zip_name = f'raray_vision_yolox_{run_name}_{TANGGAL_TRAINING}_bundle.zip'
    with zipfile.ZipFile(bundle_zip_name, 'w', zipfile.ZIP_DEFLATED) as bzf:
        if os.path.exists(best_pt_path):
            bzf.write(best_pt_path, arcname='best.pt')
        if os.path.exists(best_onnx_path):
            bzf.write(best_onnx_path, arcname='best.onnx')
        for ef in eval_target_files:
            fp = os.path.join(latest_run, ef)
            if os.path.exists(fp):
                bzf.write(fp, arcname=ef)
    print(f'✓ File Bundle Lengkap ZIP: {bundle_zip_name} ({os.path.getsize(bundle_zip_name)/(1024*1024):.2f} MB)')


### 📥 Langkah 10: Download Otomatis Bobot (`best.pt`) & Arsip Evaluasi ke Komputer
Menjalankan fungsi download Google Colab ke browser lokal Anda. Setelah terunduh, file siap diunggah ke web **Raray Vision** di menu **Model Management**.

In [ ]:
# ==========================================================
# DOWNLOAD LANGSUNG KE KOMPUTER ANDA
# ==========================================================
try:
    from google.colab import files
    print('📥 Memicu dialog pengunduhan browser...')
    
    if 'best_pt_path' in locals() and os.path.exists(best_pt_path):
        print(f'⬇️ Mengunduh model weights: best.pt ({os.path.getsize(best_pt_path)/(1024*1024):.2f} MB)...')
        files.download(best_pt_path)
    
    if 'eval_zip_name' in locals() and os.path.exists(eval_zip_name):
        print(f'⬇️ Mengunduh arsip evaluasi: {eval_zip_name}...')
        files.download(eval_zip_name)
        
    print('\n🎉 Selesai! File sedang diunduh oleh browser Anda.')
    print('\n📋 PANDUAN UNGGAH KE RARAY VISION:')
    print('1. Buka web Raray Vision > Menu "Model Management".')
    print('2. Klik tombol "+ Upload Model (.pt / .onnx)".')
    print('3. Isi formulir:')
    print('   - File Bobot Model (.pt): Pilih file best.pt yang baru diunduh.')
    print(f'   - File Arsip Evaluasi (.zip): Pilih file {eval_zip_name}.')
    print(f'   - Tanggal Training: {TANGGAL_TRAINING}')
    print(f'   - Keterangan Setting: {KETERANGAN_SETTING}')
    print('4. Klik "Simpan Model". Metrik mAP, Confusion Matrix, dan PR Curve akan langsung tampil!')
except ImportError:
    print('ℹ️ Script tidak dijalankan di Google Colab. File zip dan weights tersedia di direktori lokal.')
